In [ ]:
# =========================================
# STROKE_4.IPYNB
# ĐÁNH GIÁ MÔ HÌNH & PHÂN TÍCH KẾT QUẢ
# =========================================

# 1. IMPORT THƯ VIỆN

import pandas as pd
import numpy as np
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

# =========================================
# 2. ĐỌC DỮ LIỆU
# =========================================

df = pd.read_csv("data_da_xu_ly.csv")

X = df.drop("stroke", axis=1)
y = df["stroke"]

# Chia dữ liệu giống notebook train
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================
# 3. LOAD CÁC MODEL ĐÃ TRAIN
# =========================================

models = {
    "Decision Tree": joblib.load("model_DecisionTree.pkl"),
    "Naive Bayes": joblib.load("model_NaiveBayes.pkl"),
    "SVM": joblib.load("model_SVM.pkl"),
    "Neural Network": joblib.load("model_NeuralNetwork.pkl")
}

# =========================================
# 4. ĐÁNH GIÁ MODEL
# =========================================

results = []

for name, model in models.items():

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append([name, acc, pre, rec, f1])

    print("=" * 50)
    print(f"MODEL: {name}")
    print("=" * 50)

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

# =========================================
# 5. BẢNG SO SÁNH CÁC MODEL
# =========================================

results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score"]
)

print("\nBẢNG SO SÁNH KẾT QUẢ\n")
print(results_df)

# =========================================
# 6. VẼ CONFUSION MATRIX
# =========================================

for name, model in models.items():

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5, 4))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

    plt.show()

# =========================================
# 7. ROC CURVE & AUC
# =========================================

plt.figure(figsize=(8, 6))

for name, model in models.items():

    # Predict probability
    y_prob = model.predict_proba(X_test)[:, 1]

    # ROC
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)

    roc_auc = auc(fpr, tpr)

    # Plot
    plt.plot(
        fpr,
        tpr,
        label=f'{name} (AUC = {roc_auc:.2f})'
    )

# Đường random
plt.plot([0, 1], [0, 1], 'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve Comparison")

plt.legend()

plt.show()

# =========================================
# 8. FEATURE IMPORTANCE
# (Decision Tree)
# =========================================

dt_model = models["Decision Tree"]

importance = dt_model.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTOP FEATURE IMPORTANCE\n")
print(feature_importance)

# Vẽ biểu đồ

plt.figure(figsize=(10, 6))

sns.barplot(
    x="Importance",
    y="Feature",
    data=feature_importance.head(10)
)

plt.title("Top 10 Important Features")

plt.show()

# =========================================
# 9. ERROR ANALYSIS
# =========================================

best_model = models["Neural Network"]

y_pred_best = best_model.predict(X_test)

errors = X_test[y_test != y_pred_best]

print("\nSỐ LƯỢNG DỰ ĐOÁN SAI:")
print(len(errors))

print("\nMỘT SỐ MẪU DỰ ĐOÁN SAI:")
print(errors.head())

# =========================================
# 10. KẾT LUẬN
# =========================================

print("\nKẾT LUẬN:")

best_row = results_df.loc[
    results_df["F1-Score"].idxmax()
]

print(f"""
Mô hình tốt nhất: {best_row['Model']}

Accuracy : {best_row['Accuracy']:.4f}
Precision: {best_row['Precision']:.4f}
Recall   : {best_row['Recall']:.4f}
F1-Score : {best_row['F1-Score']:.4f}
""")